# 08 · Model Comparison & Metadata Ablation

**Purpose.** Train five architectures with two seeds each on the same split; then measure exactly how much metadata adds by retraining the best one image-only.

| | |
|---|---|
| **Reads** | `data/splits/*`, `configs/baseline.yaml` (`compare`) |
| **Writes** | `models/compare/*`, `reports/results/model_comparison*.csv`, `metadata_ablation.json`, `best_models.json` |
| **Run time (Colab T4)** | 4-8 h in total (split across teammates: finished runs are skipped) |
| **Next notebook** | 09 · Ensemble & Test Evaluation |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

In [ ]:
import time
from dermalens.evaluate import headline_metrics, predict_logits
from dermalens.model import build_model, count_parameters, zoo_entry
from dermalens.train import fit, model_from_checkpoint, train_config_from
from dermalens.utils import set_seed

TRAIN_CFG = train_config_from(CONFIG["train"])


def train_run(model_name, seed, use_meta, folder, rows=None, classes=None, weights=None, label_col="label"):
    """Train one model (or reuse it if done.json exists). Returns its validation record."""
    run_dir = paths.models / folder / f"{model_name}_{'meta' if use_meta else 'img'}_seed{seed}"
    if (run_dir / "done.json").exists():
        print("skip (done):", run_dir.name)
        return load_json(run_dir / "done.json")
    rows = rows if rows is not None else (train, val)
    classes, weights = classes or CLASSES, weights or WEIGHTS
    set_seed(seed)
    size = IMAGE_SIZE if zoo_entry(model_name)[2] == "cnn" else min(IMAGE_SIZE, 224)
    model = build_model(model_name, len(classes), CONFIG["model"]["pretrained"], use_meta,
                        CONFIG["model"]["drop_rate"], image_size=size)
    facts = {"model_name": model_name, "classes": classes, "image_size": size, "seed": seed,
             "use_metadata": use_meta}
    started = time.time()
    history = fit(model, loader(rows[0], True, size, use_meta, classes, label_col, seed),
                  loader(rows[1], False, size, use_meta, classes, label_col), weights, TRAIN_CFG, DEVICE,
                  run_dir, facts, seed=seed)
    history.to_csv(run_dir / "history.csv", index=False)
    best, _ = model_from_checkpoint(run_dir / "best.pt", DEVICE)
    logits, y_true = predict_logits(best, loader(rows[1], False, size, use_meta, classes, label_col), DEVICE)
    record = {"model": model_name, "seed": seed, "use_metadata": use_meta, "image_size": size,
              "checkpoint": str((run_dir / "best.pt").relative_to(paths.root)),
              "params_m": round(count_parameters(best)[0] / 1e6, 1),
              "minutes": round((time.time() - started) / 60, 1),
              **{k: round(float(v), 4) for k, v in headline_metrics(y_true, logits.argmax(1)).items()}}
    save_json(record, run_dir / "done.json")
    print(record)
    return record

## 1. Architectures compared

| Model | Family | Pretraining | Why it is in the comparison |
|---|---|---|---|
| EfficientNet-B0 | CNN | ImageNet-1k | v2 baseline, small and fast |
| EfficientNetV2-S | CNN | ImageNet-21k | modern efficient CNN |
| ConvNeXt-S | CNN | ImageNet-22k | strongest CNN family on small medical sets |
| Swin-T | transformer | ImageNet-1k | windowed attention, good at texture |
| ViT-B/14 DINOv2 | transformer | 142 M images, self-supervised | very general features |

Each finished run writes `done.json`; teammates can run different models on different Colab accounts.

In [ ]:
results = [train_run(name, seed, USE_META, "compare")
           for name in CONFIG["compare"]["models"] for seed in CONFIG["compare"]["seeds"]]
runs = pd.DataFrame(results)
runs.to_csv(paths.results / "model_comparison_runs.csv", index=False)
summary = runs.groupby("model").agg(
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    accuracy_mean=("accuracy", "mean"), balanced_acc_mean=("balanced_accuracy", "mean"),
    params_m=("params_m", "first"), minutes_per_run=("minutes", "mean"),
).round(4).sort_values("macro_f1_mean", ascending=False)
summary.to_csv(paths.results / "model_comparison.csv")
display(summary)
ax = summary["macro_f1_mean"].plot.bar(yerr=summary["macro_f1_std"].fillna(0), capsize=4, figsize=(8, 4),
                                        title="Validation macro-F1 (mean ± std over seeds)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(paths.figures / "model_comparison.png", dpi=150)
plt.show()

## 2. Metadata ablation: image only vs image + metadata

Same model, same seeds, same split; the only difference is the metadata branch. The difference in
macro-F1 is the contribution of age, sex and body region - a key result for the paper.

In [ ]:
best_name = summary.index[0]
image_only = pd.DataFrame([train_run(best_name, seed, False, "ablation") for seed in CONFIG["compare"]["seeds"]])
with_meta = runs[runs["model"] == best_name]
ablation = {"model": best_name,
            "image_only_macro_f1": round(float(image_only["macro_f1"].mean()), 4),
            "image_plus_metadata_macro_f1": round(float(with_meta["macro_f1"].mean()), 4),
            "image_only_accuracy": round(float(image_only["accuracy"].mean()), 4),
            "image_plus_metadata_accuracy": round(float(with_meta["accuracy"].mean()), 4)}
ablation["gain_macro_f1_points"] = round(100 * (ablation["image_plus_metadata_macro_f1"]
                                                - ablation["image_only_macro_f1"]), 1)
save_json(ablation, paths.results / "metadata_ablation.json")
display(pd.Series(ablation).to_frame("value"))

## 3. Choose the ensemble

The best run (seed) of each of the top-k architectures joins the ensemble. Different architectures
make different mistakes, so averaging them is usually 1-3 points better than the best single model.

In [ ]:
k = CONFIG["compare"]["ensemble_top_k"]
chosen = [runs[runs["model"] == name].sort_values("macro_f1", ascending=False).iloc[0].to_dict()
          for name in summary.index[:k]]
save_json({"members": chosen}, paths.results / "best_models.json")
display(pd.DataFrame(chosen)[["model", "seed", "macro_f1", "accuracy", "checkpoint"]])